# iSE Data-Lake QA — run (Kaggle, T4 x2)

Imports the `dlqa` package from your attached **code dataset** (handles either an unzipped
`dlqa/` folder or a `dlqa.zip`) and runs the full pipeline to `submission.csv`.

**Settings:** GPU T4 x2, Internet On. **Add Input:** the data lake, the dlqa code dataset, the questions .xlsx.

## 1. Dependencies

In [ ]:
!pip -q install sentence-transformers qdrant-client fastembed faster-whisper rank_bm25 python-docx tabulate rapidfuzz pymupdf beautifulsoup4 lxml openpyxl openai
!apt-get -qq install -y libreoffice >/dev/null 2>&1 && echo "libreoffice ready"

## 2. Locate `dlqa` (folder or zip) + set key

In [ ]:
import glob, os, sys, zipfile

pkg = glob.glob('/kaggle/input/**/dlqa/kaggle.py', recursive=True)
if pkg:
    PKG_DIR = os.path.dirname(os.path.dirname(pkg[0]))          # unzipped dlqa/ in the dataset
else:
    zips = glob.glob('/kaggle/input/**/dlqa.zip', recursive=True)
    assert zips, "No dlqa found — attach the dataset containing dlqa/ or dlqa.zip"
    PKG_DIR = "/kaggle/working/_dlqa_src"
    os.makedirs(PKG_DIR, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(PKG_DIR)                                    # dlqa.zip -> unzip to writable dir
sys.path.insert(0, PKG_DIR)
print("dlqa package dir:", PKG_DIR)

os.environ["DLQA_PROJECT_ROOT"] = "/kaggle/working"             # /kaggle/input is read-only
os.environ["OPENROUTER_API_KEY"] = "sk-or-REPLACE_ME"           # <-- paste your key
assert os.environ["OPENROUTER_API_KEY"] != "sk-or-REPLACE_ME", "Paste your OpenRouter key."

import dlqa.kaggle as K
print("data lake:", K.find_data_root())
print("questions:", K.find_questions() or "NOT FOUND — attach the questions .xlsx")

## 3. Build index (with image OCR) + solve everything -> submission.csv

In [ ]:
from dlqa.kaggle import run_all
run_all()